# 00 - Dataset audit (Stage 0)
**Purpose:** find out what the CICDDoS2019 files really contain *before* choosing classes, splits or caps.
Nothing is trained here. Output: `manifest.json` (schema, labels, benign counts, `Inbound` leak table, warnings).

Run order on Kaggle: Add the CICDDoS2019 dataset as input -> upload this code folder as a Kaggle Dataset and add it too -> run top to bottom.
Set `QUICK = True` first (fast look, **never feed that manifest to Stage 1**), then `QUICK = False` for the real audit.

In [ ]:
import glob, json, os, subprocess, sys
from pathlib import Path

# Folder holding s0_audit.py etc. On Kaggle: the uploaded code Dataset.
CODE_DIR = os.environ.get("NIDS_CODE") or next(
    (str(Path(p).parent) for p in glob.glob("/kaggle/input/**/s0_audit.py", recursive=True)), ".")
INPUT = os.environ.get("NIDS_INPUT", "/kaggle/input")   # where the CICDDoS2019 CSVs live
sys.path.insert(0, CODE_DIR)
from common import default_work_dir, find_files
OUT = default_work_dir()
print("code:", CODE_DIR, "| data:", INPUT, "| outputs:", OUT)

def run(script, *args):
    """Run one of our scripts and stream its output (same as `!python script args`)."""
    cmd = [sys.executable, str(Path(CODE_DIR) / script), *map(str, args)]
    print("$", " ".join(cmd))
    p = subprocess.run(cmd, capture_output=True, text=True)
    print(p.stdout[-6000:], p.stderr[-3000:])
    if p.returncode:
        raise RuntimeError(f"{script} failed (exit {p.returncode})")

## 1. Which files are attached?
We discover `.csv`/`.parquet` by extension, so folder names don't matter. File names/dates are the provenance we keep.

In [ ]:
files = find_files(INPUT)
total = 0
for p in files:
    total += p.stat().st_size
    print(f"{p.stat().st_size/1e6:10.1f} MB  {p}")
print(f"\n{len(files)} files, {total/1e9:.2f} GB")
assert files, "no csv/parquet found - did you Add Input?"

## 2. Peek at the real schema
Read 5 rows only. Look at: column names (leading spaces?), `Label` spellings, whether `Inbound`, `Timestamp`, `Protocol`, ports exist.

In [ ]:
import pandas as pd
first = files[0]
head = pd.read_parquet(first).head(5) if first.suffix == ".parquet" else pd.read_csv(first, nrows=5, skipinitialspace=True)
print(first.name, "->", head.shape[1], "columns")
print(list(head.columns))
head.iloc[:, :8]

## 3. Run the audit
`QUICK=True` reads only the first 200,000 rows of each file (the manifest is flagged as truncated). `QUICK=False` reads everything and hashes files; this is the one Stage 1 needs.

In [ ]:
QUICK = True
if QUICK:
    run("s0_audit.py", "--input", INPUT, "--out", OUT / "audit_quick", "--max-rows-per-file", 200000)
    MAN = OUT / "audit_quick" / "manifest.json"
else:
    HASH = False   # sha256 of 30 GB adds minutes; only needed for provenance in the report
    run("s0_audit.py", "--input", INPUT, "--out", OUT, *(["--hash"] if HASH else []))
    MAN = OUT / "manifest.json"

## 4. Read the verdicts
Three things decide the rest of the project: **benign count**, **Inbound leak**, **warnings**.

In [ ]:
man = json.loads(Path(MAN).read_text())
tot = man["label_totals"]
n = sum(tot.values())
print(f"scanned rows: {n:,}")
for lab, c in sorted(tot.items(), key=lambda kv: -kv[1]):
    print(f"  {lab:>10}: {c:>12,}  ({100*c/n:6.3f}%)")
b = tot.get("BENIGN", 0)
print(f"\nBENIGN raw rows: {b:,}  (distinct count comes after Stage 1 dedupe)")
if b < 20000:
    print("-> benign is scarce: plan pooled benign + capped attacks, 3 experiences max (see README).")

print("\nBenign by file:")
for p, c in man["benign_by_file"].items():
    print(f"  {c:>10,}  {Path(p).name}")

# Inbound leak test: a column whose value almost determines the class is a giveaway.
inb = man.get("inbound_by_label", {})
if inb:
    by = {}
    for k, c in inb.items():
        lab, v = k.rsplit("|", 1)
        by.setdefault(lab, {})[v] = c
    print("\nInbound value mix per label:")
    for lab, d in by.items():
        print(f"  {lab:>10}: {d}")
    dom = {l: max(d, key=d.get) for l, d in by.items()}
    purity = {l: round(max(d.values()) / sum(d.values()), 4) for l, d in by.items()}
    print("\ndominant Inbound value per label:", dom, "| purity:", purity)
    ben_dom = dom.get("BENIGN")
    split = [l for l, v in dom.items() if l != "BENIGN" and v != ben_dom]
    if ben_dom is not None and split:
        print(f"LEAK RISK: BENIGN is mostly Inbound={ben_dom} but {split} are mostly something else.")
        print("-> Inbound nearly separates benign from attacks. It is dropped by default; keep it dropped.")
    else:
        print("Inbound does not separate benign from attacks in this scan; still dropped by default (ablation only).")
else:
    print("\nNo 'inbound' column in these files.")

print("\nWARNINGS:")
print("\n".join(man["warnings"]) or "none")
print("\ntruncated:", any(f["scan_truncated"] for f in man["files"]))

## 5. What to bring back
Paste the output of section 4 plus the column list from section 2. Next: `01_splits_experiences.ipynb`.
After a `QUICK=False` run, `manifest.json` lands in `/kaggle/working/nids_out/` - keep it (Kaggle "Save Version") so you don't repeat the slow audit.